# 03. 추출 가능성 점검 — "받은 자료에서 필요한 숫자를 찾을 수 있나?"

**이 노트북이 하는 일:** 02번에서 받은 651개 보고서에서 우리가 필요한 계정(영업이익, 이자 관련, 총자산, 차입금 등 9개)을 **정말로 찾을 수 있는지**, 찾은 숫자가 **믿을 만한지** 점검합니다. **회귀나 계수는 계산하지 않습니다.** 분석할 수 있는 자료인지 보는 단계입니다.

| 항목 | 내용 |
|---|---|
| 입력 | `data/raw/DART_파일럿/fs/*.json` (02에서 받은 것) |
| 출력 | 화면 요약만 (기존 `pilot_*.csv`를 덮어쓰지 않음) |
| DART API 호출 | **없음.** 인증키 필요 없음 |
| 실행 시간 | 약 10초 |
| 먼저 알아야 할 것 | 00번(계정명 규칙·정규식), 02번(응답 한 줄의 칸) |
| 원래 스크립트 | `원본_스크립트/03_추출가능성_점검_원본.py` |

## 이 노트북의 지도
1. 파일 읽기 — 상태 분포, 연결/별도 혼재
2. 도구함 — 금액 읽기, 계정 찾기, "같은지" 비교
3. **점검 ①** 대상 계정을 얼마나 찾았나 (확보율)
4. **점검 ②** 계정 이름이 얼마나 다양한가 (변형 목록)
5. **점검 ③** 분기 값 환원이 맞아떨어지나 (항등성)
6. **점검 ④** 단위·통화
7. 추가 점검 — 금융원가는 이자와 얼마나 같은가
8. 종합: 무엇을 알게 됐고 무엇이 문제인가

> **이 점검의 기준(허용오차 0.5% 등)은 모두 임의로 정한 값입니다.** 공식·학술 기준이 아닙니다.

In [1]:
import json, re, sys
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 100)

TARGETS = config.ACCOUNT_PATTERNS                      # 00번에서 본 계정 규칙 9개
REP_BY_CODE = {v: k for k, v in config.REPORTS.items()}  # '11013' → 'Q1'
FILE_RE = re.compile(r"^(\d{8})_(\d{4})_(\d{5})_(CFS|OFS)\.json$")   # 파일 이름에서 4가지 정보를 꺼내는 규칙
print("점검할 계정:", list(TARGETS))

점검할 계정: ['assets', 'revenue', 'op_income', 'interest_exp', 'fin_cost', 'interest_paid_cf', 'st_borrow', 'cur_ltd', 'cur_bond']


---
# 1. 파일 읽기
파일 이름이 `{corp_code}_{연도}_{보고서코드}_{CFS|OFS}.json`이라서, 이름에서 **어느 기업, 어느 해, 어느 보고서, 연결/별도**인지 바로 알 수 있습니다. 위 `FILE_RE`가 그 4가지를 꺼내는 규칙입니다.

In [2]:
files = []
for path in sorted((config.RAW / "fs").glob("*.json")):
    m = FILE_RE.match(path.name)
    if not m:
        continue
    corp, year, rep_code, fs = m.groups()
    files.append({
        "corp": corp, "year": int(year), "rep": REP_BY_CODE.get(rep_code, rep_code), "fs": fs,
        "data": json.loads(path.read_text(encoding="utf-8")),
    })

ok_files = [f for f in files if f["data"].get("status") == "000"]
print(f"파일 {len(files):,}개 중 자료가 있는 것(상태 000): {len(ok_files)}개")
print("상태 분포:", dict(Counter(f["data"].get("status") for f in files)))
print("연결/별도 분포(자료가 있는 파일):", dict(Counter(f["fs"] for f in ok_files)))

파일 1,768개 중 자료가 있는 것(상태 000): 651개
상태 분포: {'013': 1117, '000': 651}
연결/별도 분포(자료가 있는 파일): {'OFS': 179, 'CFS': 472}


### 해석
- 1,768개 파일 중 **자료가 있는 것이 651개**입니다(나머지는 "조회 데이터 없음"). 이후 점검은 이 651개를 대상으로 합니다.
- 연결(CFS) 472개, 별도(OFS) 179개입니다.

### 연결·별도가 섞였는가? (규칙을 어긴 사례 찾기)
우리 규칙은 "연결이 있으면 연결, 없으면 별도"입니다. 그런데 **한 기업 안에서 어떤 해는 연결, 어떤 해는 별도**가 되면 **같은 기업의 숫자가 중간에 끊깁니다**(연결은 자회사까지 합친 규모라 값이 크게 다를 수 있음).

In [3]:
fs_by_year, fs_by_firm = defaultdict(set), defaultdict(set)
for f in ok_files:
    fs_by_year[(f["corp"], f["year"])].add(f["fs"])
    fs_by_firm[f["corp"]].add(f["fs"])

mixed_year = {k: v for k, v in fs_by_year.items() if len(v) > 1}
mixed_firm = {k: v for k, v in fs_by_firm.items() if len(v) > 1}
print(f"같은 기업-연도 안에서 연결·별도가 섞인 건수: {len(mixed_year)}")
print(f"같은 기업이 연도에 따라 연결·별도가 달라진 기업 수: {len(mixed_firm)}")

meta = pd.read_csv(config.OUT / "파일럿_표본40곳.csv", dtype=str, keep_default_na=False).set_index("corp_code")
for c in mixed_firm:
    yrs = sorted({(f["year"], f["fs"]) for f in ok_files if f["corp"] == c})
    by = defaultdict(list)
    for y, fs in yrs:
        by[fs].append(y)
    print(f"  - {meta.loc[c, 'corp_name']}({c}): 연결 {by.get('CFS', [])} / 별도 {by.get('OFS', [])}")

같은 기업-연도 안에서 연결·별도가 섞인 건수: 1
같은 기업이 연도에 따라 연결·별도가 달라진 기업 수: 3
  - 차AI헬스케어(00136776): 연결 [2018, 2019, 2020, 2021, 2022, 2023, 2024] / 별도 [2018]
  - 에이디칩스(00356927): 연결 [2022, 2023, 2024] / 별도 [2018, 2019, 2020, 2021]
  - 미쥬(01547933): 연결 [2024] / 별도 [2022, 2023]


### 해석
- 규칙을 어긴 기업이 **3곳 있습니다**(위 출력): 차AI헬스케어는 2018년만 별도·나머지는 연결, 에이디칩스는 2018~2021년 별도 → 2022년부터 연결, 미쥬는 2022~2023년 별도 → 2024년 연결입니다. 이유는 "연결 우선, 없으면 별도"라는 **규칙이 해마다 따로 적용되기 때문**입니다. 어느 해에 연결 재무제표가 처음 생기거나(자회사 취득 등으로 추정, 확인은 안 함) 사라지면 기준이 바뀝니다.
- 이런 기업은 **연결과 별도를 한 기업 안에서 섞지 않는 처리 규칙**(예: 한 기업은 한 가지 기준으로 통일, 못 하면 제외)이 필요합니다. 아직 정하지 않았습니다(사전 고정 문서 3절).
- 28곳 중 3곳(약 11%)입니다. 표본이 작고 대표성이 없어서 이 비율을 전체에 적용하면 안 되지만, **무시할 수 있는 수준이 아니라는 신호**로는 볼 수 있습니다.
- **한 기업 안에서 기준이 바뀌는 지점은 ICR 분모·분자의 "규모"가 갑자기 바뀌는 지점**이라 FE(기업 고정효과)로도 완전히 처리되지 않을 수 있습니다. 이런 기업의 처리 방식(통일 / 구간별 분리 / 제외)을 정해야 합니다.

---
# 2. 도구함 — 세 가지 작은 함수

In [4]:
def parse_amount(text):
    # DART 금액은 글자('1,234,567')로 온다. 숫자로 바꾼다. 비었거나 '-'이면 None(= 값 없음)
    if text is None:
        return None
    s = str(text).replace(",", "").strip()
    if s in ("", "-"):
        return None
    try:
        return float(s)
    except ValueError:
        return None


def first_match(rows, sj_pattern, name_pattern):
    # 계정 목록(rows)에서 '표 구분'과 '계정명 규칙'에 맞는 줄을 찾는다. (찾은 줄, 맞은 줄 수)를 돌려준다
    sj_re, nm_re = re.compile(sj_pattern), re.compile(name_pattern)
    matches = [
        r for r in rows
        if sj_re.fullmatch(r.get("sj_div", "")) and nm_re.search(re.sub(r"\s+", "", r.get("account_nm", "")))
    ]
    if not matches:
        return None, 0
    def order_key(r):
        try:
            return int(r.get("ord", "0") or 0)
        except ValueError:
            return 0
    matches.sort(key=order_key)           # 같은 규칙에 여러 줄이 걸리면 ord(표 안 순서)가 가장 앞인 줄을 쓴다
    return matches[0], len(matches)


TOL_REL = 0.005   # 허용오차 0.5% (임의 기준)

def check_close(a, b):
    # a와 b가 같은가? 'exact'(정확히) / 'within_tol'(0.5% 이내) / 'exceed'(0.5% 초과)
    if abs(a - b) < 1e-9:
        return "exact"
    scale = max(abs(a), abs(b), 1.0)
    return "within_tol" if abs(a - b) <= TOL_REL * scale else "exceed"


print("parse_amount('1,234,567') =", parse_amount("1,234,567"))
print("parse_amount('-')         =", parse_amount("-"))
print("parse_amount(None)        =", parse_amount(None))
print("check_close(100, 100)     =", check_close(100, 100))
print("check_close(100, 100.4)   =", check_close(100, 100.4), " (차이 0.4% → 허용오차 안)")
print("check_close(100, 103)     =", check_close(100, 103), "(차이 3% → 초과)")

parse_amount('1,234,567') = 1234567.0
parse_amount('-')         = None
parse_amount(None)        = None
check_close(100, 100)     = exact
check_close(100, 100.4)   = within_tol  (차이 0.4% → 허용오차 안)
check_close(100, 103)     = exceed (차이 3% → 초과)


### 해석
- `parse_amount`: 금액은 숫자가 아니라 글자로 옵니다. 쉼표를 지우고 숫자로 바꿉니다. **비어 있는 값은 0이 아니라 `None`(없음)** 입니다. 0과 "없음"을 구분하는 것이 중요합니다.
- `first_match`: 같은 규칙에 **여러 줄이 걸릴 수 있습니다**(예: 포괄손익계산서에 '영업이익'과 '계속영업이익'). 그래서 맞은 줄 수도 함께 돌려주고, 여러 줄이면 `ord`(표 안 순서)가 앞선 줄을 씁니다. **이 선택이 항상 맞는지는 검증하지 않았습니다.**
- `check_close`: 두 숫자가 같은지 볼 때 **허용오차**가 필요한 이유는 기업이 반올림해서 공시하는 경우가 있어서입니다. 0.5%는 임의로 정한 값입니다.

---
# 3. 점검 ① — 필요한 계정을 얼마나 찾았나 (확보율)
651개 보고서 각각에서 9개 계정을 규칙으로 찾아봅니다. 확보율 = (찾은 보고서 수) ÷ 651.

In [5]:
cover = defaultdict(Counter)                       # 계정별로, 보고서 종류별로 찾은 수
n_by_rep = Counter(f["rep"] for f in ok_files)
store = {}                                          # (기업, 연도, 연결/별도, 보고서) → {계정: 찾은 값}
variants = Counter()                                # 계정 이름 변형 세기
currencies = Counter()
asset_values = defaultdict(list)
VARIANT_PATTERN = re.compile(r"차입|사채|이자|금융비용|금융원가|영업이익|유동성|장기부채")

for f in ok_files:
    rows = f["data"].get("list", [])
    entry = {}
    for r in rows:
        currencies[r.get("currency", "")] += 1
        if VARIANT_PATTERN.search(r.get("account_nm", "")):
            variants[(r.get("sj_div", ""), r.get("account_nm", ""))] += 1
    for tname, (sj, pat) in TARGETS.items():
        row, n_hit = first_match(rows, sj, pat)
        if row is None:
            continue
        cover[tname][f["rep"]] += 1
        amt = parse_amount(row.get("thstrm_amount"))
        add = parse_amount(row.get("thstrm_add_amount"))
        entry[tname] = {"amount": amt, "cum": add, "n_matches": n_hit, "name": row.get("account_nm")}
        if tname == "assets" and amt is not None:
            asset_values[f["corp"]].append(amt)
    store[(f["corp"], f["year"], f["fs"], f["rep"])] = entry

rows_out = []
for tname in TARGETS:
    total = sum(cover[tname][rep] for rep in config.REPORTS)
    rows_out.append({"계정": tname, "찾은 보고서 수": total, "확보율(%)": round(100 * total / len(ok_files), 1)})
cov = pd.DataFrame(rows_out).set_index("계정")

INTEREST_PROXIES = ("interest_exp", "fin_cost", "interest_paid_cf")
any_int = sum(1 for e in store.values() if any(t in e for t in INTEREST_PROXIES))
cov.loc["(이자 대용 중 하나라도)"] = [any_int, round(100 * any_int / len(store), 1)]
cov

,찾은 보고서 수,확보율(%)
계정,,
assets,650.0,99.8
revenue,607.0,93.2
op_income,651.0,100.0
interest_exp,1.0,0.2
fin_cost,648.0,99.5
interest_paid_cf,625.0,96.0
st_borrow,471.0,72.4
cur_ltd,261.0,40.1
cur_bond,19.0,2.9


### 해석
| 계정 | 확보율 | 읽는 법 |
|---|---|---|
| 총자산·영업이익 | 약 100% | 거의 모든 보고서에서 찾음. 문제 없음 |
| 매출액 | 93% | 일부 보고서에서 못 찾음. 이유는 확인하지 못함(기업별 표에서 매출을 6~12개 보고서에서만 찾은 기업이 있음) |
| **이자비용(`interest_exp`)** | **0.2%** | 사실상 **없음**. 손익계산서에 '이자비용'이라는 이름의 계정을 따로 두는 기업이 거의 없음 |
| **금융원가·금융비용(`fin_cost`)** | 99.5% | 거의 다 있음 → ICR 분모의 **대용품 후보** |
| 현금흐름표 이자지급 | 96% | 있음 (누적 값으로 보임) |
| 단기차입금 | 72% | **28%는 못 찾음** — 진짜 없는지, 이름이 달라서인지 구분 안 됨 |
| 유동성장기차입금 | 40% | 절반 넘게 못 찾음 |
| 유동성사채 | 3% | 거의 없음 (사채를 발행한 기업이 적거나 이름이 다름) |

핵심 문제 두 가지:
1. **ICR의 분모(이자비용)를 직접 구할 수 없습니다.** 금융원가로 대신해야 하는데 금융원가는 이자비용이 아닙니다(7절에서 얼마나 다른지 봅니다).
2. **노출도(E)의 재료인 차입금 계정은 "없음"이 두 가지 뜻입니다.** 정말 단기차입금이 0원이라서 응답에 안 나온 것일 수도 있고, 있는데 이름이 달라서 우리 규칙이 못 잡은 것일 수도 있습니다. 0으로 처리하면 E가 과소 계산될 수 있습니다(결정 A10, A15).
- 위 "이자 대용 하나라도"는 **이자비용·금융원가·이자지급 중 하나라도 찾은 비율**이며, 이 셋은 서로 다른 개념이라 "찾았다"가 "이자를 구했다"는 뜻은 아닙니다.

### 기업 단위로 다시 보기
실패가 **일부 기업만의 문제**인지 **전반적 규칙 문제**인지 구분하는 것이 파일럿 통과 판단의 기준입니다(사전 고정 문서 9절: "수정 가능한 규칙 문제인가, 구조적으로 못 구하는 항목인가"). 28곳 각각이 계정을 몇 번 찾았는지 봅니다.

In [6]:
firm_rows = []
for corp in sorted({k[0] for k in store}):
    entries = [e for k, e in store.items() if k[0] == corp]
    row = {"기업": meta.loc[corp, "corp_name"], "보고서 수": len(entries)}
    for tname in TARGETS:
        row[tname] = sum(1 for e in entries if tname in e)
    row["이자 대용 있음"] = sum(1 for e in entries if any(t in e for t in INTEREST_PROXIES))
    firm_rows.append(row)
firm_cov = pd.DataFrame(firm_rows)
firm_cov

,기업,보고서 수,assets,revenue,op_income,interest_exp,fin_cost,interest_paid_cf,st_borrow,cur_ltd,cur_bond,이자 대용 있음
0,대륙제관,28,28,28,28,0,28,28,22,21,0,28
1,디와이덕양,28,28,28,28,0,28,28,27,27,0,28
2,TYM,28,28,28,28,0,28,28,24,0,0,28
3,삼보산업,28,28,28,28,0,28,28,22,22,0,28
4,삼화전기,28,28,28,28,0,28,28,27,0,0,28
5,차AI헬스케어,28,28,28,28,0,28,28,27,17,0,28
6,한탑,28,28,28,28,0,28,19,0,0,0,28
7,영보화학,28,28,28,28,0,28,28,22,13,0,28
8,에이프로젠,28,28,12,28,0,28,28,28,6,0,28
9,한일홀딩스,28,28,6,28,0,28,28,28,28,0,28


In [7]:
no_int = firm_cov[firm_cov["이자 대용 있음"] == 0]["기업"].tolist()
no_borrow = firm_cov[(firm_cov["st_borrow"] == 0) & (firm_cov["cur_ltd"] == 0) & (firm_cov["cur_bond"] == 0)]["기업"].tolist()
print(f"이자 대용(이자비용·금융원가·이자지급)이 한 번도 안 구해진 기업: {len(no_int)}곳 {no_int}")
print(f"차입 세 항목(단기차입금·유동성장기차입금·유동성사채)이 모두 한 번도 안 구해진 기업: {len(no_borrow)}곳 {no_borrow}")

이자 대용(이자비용·금융원가·이자지급)이 한 번도 안 구해진 기업: 1곳 ['미쥬']
차입 세 항목(단기차입금·유동성장기차입금·유동성사채)이 모두 한 번도 안 구해진 기업: 5곳 ['한탑', '에스앤디', '마녀공장', '세니젠', '미쥬']


### 해석
- **숫자는 "그 기업의 보고서 N개 중 몇 개에서 찾았나"** 입니다. 예를 들어 `st_borrow`가 22/28이면 28개 보고서 중 22개에서 단기차입금을 찾았다는 뜻입니다. 22/28처럼 **일부만 찾는 기업**은 "이 기업은 단기차입금이 아예 없다"가 아니라 **해마다 계정 이름이 바뀌었거나 값이 0인 해가 있다**는 뜻일 수 있습니다.
- 이자 대용이 전혀 안 구해진 기업은 **미쥬 1곳뿐**입니다. 즉 이자 문제는 "특정 기업의 문제"가 아니라 **"이자비용이라는 이름 자체가 거의 없다"는 전반적 구조 문제**입니다(모든 기업에서 `interest_exp` 열이 거의 0).
- **차입 세 항목이 한 번도 안 구해진 기업이 5곳**(한탑, 에스앤디, 마녀공장, 세니젠, 미쥬)입니다. 정말 차입이 없는 기업인지, 이름이 달라서 못 잡은 것인지 **이 표만으로는 구분할 수 없습니다.** 이전 메모에는 "9곳은 차입 항목이 전혀 없음"이라고 적혀 있는데 이 노트북의 5곳과 숫자가 다릅니다. 두 수치의 정의(어떤 규칙으로, 어떤 단위로 센 것인지)가 달랐을 수 있으나 **확인하지 못했습니다.** 둘을 같은 것으로 취급하지 마세요.
- 한탑처럼 **보고서 28개 중 28개에서 총자산·영업이익은 찾는데 차입 항목은 한 번도 못 찾은** 기업은, 사실상 무차입일 수도 있고 규칙에 없는 이름을 쓸 수도 있습니다. 이런 기업을 E=0으로 둘지 "정의 불가"로 둘지가 A10입니다.

---
# 4. 점검 ② — 계정 이름은 얼마나 다양한가 (변형 목록)
규칙이 못 잡는 이름을 찾으려면 **실제로 나온 이름을 모아 보는** 수밖에 없습니다. 차입·사채·이자·금융비용·영업이익·유동성·장기부채 관련 이름만 모아 빈도를 셉니다.

In [8]:
var = pd.DataFrame(
    [{"표": k[0], "계정명": k[1], "나온 횟수": v} for k, v in variants.most_common()]
)
print(f"관련 계정 이름의 종류: {len(var)}가지")
var.head(30)

관련 계정 이름의 종류: 220가지


,표,계정명,나온 횟수
0,CIS,영업이익(손실),502
1,BS,단기차입금,469
2,CF,단기차입금의 상환,368
3,CF,단기차입금의 증가,356
4,BS,장기차입금,333
5,CIS,금융비용,310
6,CF,이자수취,308
7,CIS,금융원가,282
8,CF,이자지급(영업),279
9,CF,이자지급,233


### 해석
- 이름이 **200가지 넘게** 나옵니다(위 출력). 같은 뜻인데 표기만 다른 것들이 많습니다.
  - 이자지급: '이자지급', '이자지급(영업)', '이자의 지급'
  - 금융비용: '금융비용', '금융원가'
  - 영업이익: '영업이익', '영업이익(손실)'
- 이 목록 안에는 **우리가 원하지 않는 이름**도 섞여 있습니다(예: 현금흐름표의 '단기차입금의 상환', 자본변동표의 '전환사채의 전환'). 그래서 규칙에 **표 구분(sj_div)을 함께 걸어** 엉뚱한 표의 이름이 잡히지 않게 한 것입니다.
- 규칙(`ACCOUNT_PATTERNS`)에 아직 반영하지 않은 변형이 있습니다(예: BS의 '유동 차입금', '유동성전환사채'). 이것을 반영할지가 결정 **A15**입니다. 반영하면 커버리지가 올라가지만(이전 점검에서 기업 단위 86.5%), **'전환사채'를 유동성사채와 같은 것으로 볼지** 같은 **판단이 들어가는 매핑**이라 기계적으로 넓히면 안 됩니다.

---
# 5. 점검 ③ — 분기 값 환원이 맞아떨어지나 (항등성)

**왜 점검하나?** 우리가 쓸 값은 **분기별 영업이익·이자**입니다. 그런데 DART는 분기 값(3개월)과 누적 값을 따로 줍니다. 두 가지가 **서로 모순 없이 맞는지**를 확인해야 합니다. 맞아야 하는 식은 두 개입니다.

1. **Q1 + Q2 = 반기 누적** (1분기 3개월 값 + 2분기 3개월 값 = 1~6월 누적)
2. **반기 누적 + Q3 = 3분기 누적**

(Q2 값은 반기 보고서의 3개월 값, Q3 값은 3분기 보고서의 3개월 값입니다.) 맞지 않으면 **정정공시, 기업 내부 재분류, 우리 쪽 읽기 오류** 중 하나입니다. 이유까지는 이 점검으로 알 수 없습니다.

In [9]:
by_firm_year = defaultdict(dict)
for (corp, year, fs, rep), entry in store.items():
    by_firm_year[(corp, year, fs)][rep] = entry

q_rows = []
for (corp, year, fs), reps in sorted(by_firm_year.items()):
    for tname in ("op_income", "interest_exp", "fin_cost"):
        q1, h1, q3 = (reps.get(k, {}).get(tname) for k in ("Q1", "H1", "Q3"))
        res = {"corp": corp, "year": year, "fs": fs, "target": tname,
               "Q1+Q2=H1누적": "", "H1누적+Q3=Q3누적": ""}
        if q1 and h1 and None not in (q1["amount"], h1["amount"], h1["cum"]):
            res["Q1+Q2=H1누적"] = check_close(q1["amount"] + h1["amount"], h1["cum"])
            res["_q1"], res["_h1_3m"], res["_h1_cum"] = q1["amount"], h1["amount"], h1["cum"]
        if h1 and q3 and None not in (h1["cum"], q3["amount"], q3["cum"]):
            res["H1누적+Q3=Q3누적"] = check_close(h1["cum"] + q3["amount"], q3["cum"])
        q_rows.append(res)
qdf = pd.DataFrame(q_rows)

summary = []
for tname in ("op_income", "interest_exp", "fin_cost"):
    for col in ("Q1+Q2=H1누적", "H1누적+Q3=Q3누적"):
        vals = qdf[(qdf["target"] == tname) & (qdf[col] != "")][col]
        c = Counter(vals)
        summary.append({"계정": tname, "식": col, "정확히 같음": c["exact"], "0.5% 이내": c["within_tol"],
                        "0.5% 초과": c["exceed"], "검사한 건수": len(vals)})
pd.DataFrame(summary)

,계정,식,정확히 같음,0.5% 이내,0.5% 초과,검사한 건수
0,op_income,Q1+Q2=H1누적,153,1,4,158
1,op_income,H1누적+Q3=Q3누적,152,2,6,160
2,interest_exp,Q1+Q2=H1누적,0,0,0,0
3,interest_exp,H1누적+Q3=Q3누적,0,0,0,0
4,fin_cost,Q1+Q2=H1누적,140,5,13,158
5,fin_cost,H1누적+Q3=Q3누적,147,5,8,160


### 해석
- **영업이익**: 첫 식 158건 중 153건 정확히 일치·1건 허용오차 이내·**4건 초과**, 둘째 식 160건 중 152·2·**6건 초과**입니다. 약 97% 이상 맞습니다.
- **금융원가**: 첫 식 158건 중 140·5·**13건 초과**, 둘째 식 160건 중 147·5·**8건 초과**로, 영업이익보다 **어긋나는 비율이 높습니다**(약 5~8%).
- **이자비용**은 계정 자체가 거의 없어 검사 건수가 **0**입니다.
- 어긋나는 건(0.5% 초과)이 어떻게 생겼는지 영업이익 4건을 직접 봅니다.

In [10]:
bad = qdf[(qdf["target"] == "op_income") & (qdf["Q1+Q2=H1누적"] == "exceed")].copy()
bad["기업"] = bad["corp"].map(meta["corp_name"])
bad["Q1+Q2 (우리 계산)"] = bad["_q1"] + bad["_h1_3m"]
bad["반기 누적 (원문)"] = bad["_h1_cum"]
bad["차이(%)"] = ((bad["Q1+Q2 (우리 계산)"] - bad["반기 누적 (원문)"]) / bad["반기 누적 (원문)"].abs() * 100).round(1)
bad[["기업", "year", "fs", "_q1", "_h1_3m", "Q1+Q2 (우리 계산)", "반기 누적 (원문)", "차이(%)"]].rename(
    columns={"_q1": "Q1(3개월)", "_h1_3m": "반기 보고서의 3개월"})

,기업,year,fs,Q1(3개월),반기 보고서의 3개월,Q1+Q2 (우리 계산),반기 누적 (원문),차이(%)
192,한일홀딩스,2018,CFS,-6.363663e+09,2.440626e+10,1.804259e+10,1.198868e+10,50.5
258,수성웹툰,2019,CFS,-3.928637e+08,5.728598e+07,-3.355777e+08,4.494799e+08,-174.7
330,에이디칩스,2022,CFS,4.906257e+08,-1.101316e+09,-6.106904e+08,-6.069404e+08,-0.6
336,에이디칩스,2024,CFS,-5.195012e+09,-6.083832e+09,-1.127884e+10,-7.144112e+09,-57.9


### 해석
- 4건 중 **세 건은 차이가 매우 큽니다**(+50.5%, −174.7%, −57.9%). 특히 수성웹툰 2019년은 우리가 계산한 합(Q1+Q2)이 **음수(−3.4억)인데 반기 누적은 양수(+4.5억)** 라서 부호까지 다릅니다. 한 건(에이디칩스 2022, −0.6%)은 허용오차 0.5%를 살짝 넘긴 경계 사례입니다.
- 이런 모순은 분기 값과 누적 값이 서로 다른 **기준(재분류, 정정, 연결 범위 변화)** 에서 나왔을 가능성이 있다고 **의심**하지만, **원인은 확인하지 못했습니다.** 원 공시와 대조해야 알 수 있습니다.
- 이전 문서의 "3~8% 어긋나는 경우가 있음"과 이번 4건의 범위가 다릅니다. 같은 점검을 다른 식(예: 다른 계정·다른 허용오차)으로 했을 가능성이 있으나 확인하지 못했습니다.
- 이 어긋남은 **분기 값 환원의 오차**로 이어집니다. 우리는 4분기를 "연간 − 3분기 누적"으로 계산하므로, 3분기 누적 값이 어긋나 있으면 **4분기 값 하나에 그 오차가 몰려서** 나옵니다.
- 현재 허용오차 0.5%는 임의입니다. 이 기준을 바꾸면 "통과/탈락" 수가 바뀌는 종류의 값이라 **기준을 먼저 정하고 결과를 보는** 순서가 중요합니다.

---
# 6. 점검 ④ — 단위와 통화

In [11]:
print("[통화 값]", dict(currencies))
meds = sorted(sum(v) / len(v) for v in asset_values.values())
print(f"[총자산 값의 규모(기업별 평균)] 최소 {meds[0]:,.0f} / 중앙 {meds[len(meds)//2]:,.0f} / 최대 {meds[-1]:,.0f}")
print(f"  → 중앙값이 약 {meds[len(meds)//2]/1e8:,.0f}억 원 규모")

[통화 값] {'KRW': 100937}
[총자산 값의 규모(기업별 평균)] 최소 16,590,721,216 / 중앙 148,149,365,010 / 최대 11,732,269,867,063
  → 중앙값이 약 1,481억 원 규모


### 해석
- 통화는 **모두 KRW(원)** 입니다.
- 총자산이 최소 약 166억 원, 중앙 약 1,481억 원, 최대 약 11.7조 원입니다. 상장 제조업체 규모로 자연스러워서 **원 단위로 읽고 있다**는 것과 일치합니다. 천원·백만원 단위였다면 값이 훨씬 작게 나왔을 것입니다.
- 다만 "단위가 기업마다 같은가"는 **응답 한 건 기준으로 원 단위로 보인다**는 관찰일 뿐 전수 확인이 아닙니다. 값이 지나치게 작은 기업이 있는지만 확인했습니다.

---
# 7. 추가 점검 — 금융원가는 이자와 얼마나 같은가

**왜 중요한가?** 이자비용을 직접 구할 수 없어서 **금융원가(금융비용)** 를 대신 쓰려고 합니다. 그런데 금융원가에는 이자 외에도 **외환손실, 파생상품손실, 대손 관련 비용 등**이 섞일 수 있습니다. 이것이 크면 "ICR(이자보상비율)"이 아니라 "금융비용 보상비율"이 됩니다.

**어떻게 비교하나?** 현금흐름표의 **이자지급**(실제로 이자로 나간 현금)을 이자의 또 다른 대용품으로 두고, 같은 해의 금융원가와 비율을 봅니다.

비율 = |이자지급| ÷ 금융원가 (사업보고서 기준, 연간 값끼리)

- 비율이 **1에 가까우면** 두 지표가 비슷하다는 뜻
- **작으면(예: 0.4)** 금융원가가 이자지급보다 훨씬 크다는 뜻 → 금융원가에 이자 아닌 비용이 많이 섞였거나, 이자를 현금이 아닌 방식(만기 도래 전 발생분 등)으로 처리했을 가능성

> 이 비교는 **이자 정의 문제의 규모를 가늠하는 데만** 쓰입니다. 어느 쪽이 "진짜 이자"에 가까운지는 알려주지 않습니다.

In [12]:
ratio_rows = []
for (corp, year, fs, rep), e in store.items():
    if rep != "FY":
        continue
    fc, ip = e.get("fin_cost"), e.get("interest_paid_cf")
    if fc and ip and fc["amount"] and ip["amount"] and fc["amount"] > 0:
        ratio_rows.append({"corp": corp, "year": year, "fs": fs,
                           "fin_cost": fc["amount"], "interest_paid": abs(ip["amount"]),
                           "ratio": abs(ip["amount"]) / fc["amount"]})
rdf = pd.DataFrame(ratio_rows)
print(f"비교 가능한 기업-연도: {len(rdf)}건 (기업 {rdf['corp'].nunique()}곳)")
print()
q = rdf["ratio"].quantile([0.1, 0.25, 0.5, 0.75, 0.9]).round(2)
print("|이자지급| ÷ 금융원가 분포 (분위수):")
print(q.to_string())
print()
bins = pd.cut(rdf["ratio"], [0, 0.5, 0.8, 1.2, 2, np.inf], labels=["<0.5", "0.5~0.8", "0.8~1.2", "1.2~2", ">2"])
print("구간별 건수:")
print(bins.value_counts().sort_index().to_string())

비교 가능한 기업-연도: 156건 (기업 27곳)

|이자지급| ÷ 금융원가 분포 (분위수):
0.10    0.03
0.25    0.12
0.50    0.38
0.75    0.81
0.90    0.99

구간별 건수:
ratio
<0.5       90
0.5~0.8    26
0.8~1.2    37
1.2~2       3
>2          0


### 해석
- 비율의 **중앙값은 약 0.38**입니다(이전 메모의 0.39와 거의 같고, 비교 가능한 건수가 156건으로 메모의 158건과 약간 달라서 생긴 차이로 보이나 확인하지 못했습니다). **156건 중 90건(58%)이 0.5 미만**이고, 1.2를 넘는 것은 3건뿐입니다. 즉 **대부분의 기업에서 이자지급이 금융원가보다 작습니다.**
- 분포가 **넓게 퍼져 있습니다**(25%분위 0.12, 75%분위 0.81). 기업마다 두 값의 관계가 크게 다르다는 뜻이고, **금융원가를 이자의 대용품으로 쓰면 기업 간 비교에 오차가 체계적으로 들어갈 수 있습니다.**
- **다만 비율이 낮은 이유가 "금융원가에 이자 아닌 비용이 섞여서"라고 단정할 수는 없습니다.** 가능한 설명을 모두 적습니다(어느 것도 검증하지 못했습니다).
  1. 금융원가에 외환손실·파생손실 등이 섞임 (우리가 의심하는 설명)
  2. 이자지급이 **영업활동이 아닌 재무활동 현금흐름**에 적혀 있거나, 여러 줄로 나뉘어 있는데 우리 코드는 **규칙에 맞는 첫 줄만** 읽음 → 이자지급을 **과소**로 잡았을 수 있음
  3. 이자를 **발생한 해와 지급한 해가 다름**(발생주의 vs 현금주의), 자본화된 이자
  4. 현금흐름표가 누적 값이라는 우리의 해석이 일부 기업에서 틀림
- **이 오차가 단순한 잡음이 아닐 수 있는 이유:** 금융원가에 섞인 **외환손익은 환율에 따라 움직입니다.** 2022년처럼 환율이 급등한 해에는 외화부채가 많은 기업의 금융원가가 커집니다. 그 기업들이 우리가 만든 노출도(단기차입 비중)와도 연관돼 있다면, **금리 효과가 아닌 환차손이 "노출도 효과"로 읽힐 위험**이 있습니다. 이 가능성은 **아직 검증하지 못했습니다**(기업별 외화부채 정보가 필요하고, 주석에 있음).
- 이 표본은 28곳이라 **대표성이 없습니다**(위 건수). 중앙값 0.4를 전체의 값으로 말하면 안 됩니다.
- 이 점검이 결정 **A3·A16**(이자 정의)의 핵심 근거입니다. 현재 제안은 "약 10곳의 **주석**(이자비용이 따로 적혀 있는 부분)과 대조해서 금융원가가 이자에 얼마나 가까운지 보고 정한다"인데, **아직 하지 않았습니다**(허락 필요).

---
# 8. 종합: 점검 결과가 말해 주는 것

| 점검 | 결과 | 의미 | 관련 결정 |
|---|---|---|---|
| ① 총자산·영업이익 | ≈100% 확보 | 추출 가능 | — |
| ① **이자비용** | **0.2%** | **직접 구할 수 없음 (구조적)** | A3, A16 |
| ① 금융원가 / CF 이자지급 | 99.5% / 96% | 대용품 후보 존재. 다만 서로 다름 | A3, A16 |
| ① 차입 3항목 | 72% / 40% / 3% | 없음의 두 뜻(진짜 0 vs 이름 불일치) 구분 불가 | A10, A15 |
| ② 계정 이름 | 200여 가지 | 규칙 확장 필요, 판단 개입 | A15 |
| ③ 분기 환원 | 대부분 일치, 일부 불일치 | 일부 기업-연도는 분기 값 신뢰도 낮음 | A13(정정공시) |
| ③ 연결/별도 혼재 | 3개 기업 | 처리 규칙 필요 | 사전 고정 문서 3절 |
| ④ 단위 | KRW, 원 단위로 보임 | 이상 없음(전수 확인은 아님) | — |
| 7 금융원가 vs 이자지급 | 비율 중앙값 약 0.38, 58%가 0.5 미만 | 두 지표가 크게 다름. 원인은 미확인(위 4가지 가설) | A3, A16 |

### 이 점검의 한계
- **28곳(자료가 있는 곳)뿐**이라 비율을 일반화할 수 없습니다. 이 결과로 "A 전체가 가능하다/불가능하다"를 판정하면 안 됩니다. 사전 고정 문서도 "작은 표본의 우연한 성공·실패에 주제 전체를 맡기지 않는다"고 적었습니다.
- **통과/실패 기준(확보율 몇 % 이상 등)은 정해진 것이 없습니다.** 이 노트북은 사실만 보여 주며 통과 판정을 하지 않습니다.
- 정정공시, 주석, 일괄 다운로드는 확인하지 않았습니다.

### 다음 단계
`04_정제표_예시.ipynb`: 원자료를 "기업 × 분기" 표로 바꿔 보고, TTM·ICR·차입비중이 어떻게 계산되는지 한 기업으로 따라가 봅니다.